In [ ]:
"""

This module provides comprehensive visualization capabilities for outlier detection
datasets using multiple dimensionality reduction techniques and density analysis.

Key Features:
  - Distance matrix calculation and density estimation
  - Dimensionality reduction (PCA, t-SNE, UMAP)
  - 2D and 3D scatter plot visualization
  - Image processing and composition
  - Statistical density analysis and reporting

Organization:
  1. Installation & Dependencies
  2. Configuration & Constants
  3. Data Processing Functions
  4. Density Analysis Functions
  5. Dimensionality Reduction Functions
  6. Visualization Functions
  7. Image Processing Functions
  8. Main Execution Pipeline
"""

import os
import re
import math
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from typing import Dict, List, Tuple
import warnings

from scipy.spatial.distance import cdist, pdist
from tqdm.notebook import tqdm
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from umap import UMAP

# Suppress all library warnings for cleaner output
warnings.filterwarnings("ignore")

# Configure pandas display options
pd.set_option('display.max_rows', 50)

In [ ]:
# =============================================================================
# SECTION 1: Configuration & Constants
# =============================================================================

# Available dimensionality reduction methods
PROJECTION_METHODS = [PCA, TSNE, UMAP]

# Density radius percentages for analysis
DENSITY_RADII = [0.01, 0.05, 0.10, 0.20, 0.25]
DENSITY_RADIUS_LABELS = ['R1%', 'R5%', 'R10%', 'R20%', 'R25%']

# Projection dimensions for visualization
DEFAULT_2D_DIMENSIONS = 2
DEFAULT_3D_DIMENSIONS = 3

# Image processing parameters
DEFAULT_IMAGE_WIDTH = 1310
DEFAULT_IMAGE_HEIGHT = 630
FIGURE_SCALE = 3

# Crop coordinates for different projection dimensions
CROP_COORDINATES = {
    '0_0': ((190, 815), (120, 1468)),
    '0_1': ((190, 825), (120, 1468)),
    '1_0': ((190, 815), (190, 1468)),
    '1_1': ((190, 825), (190, 1468)),
}

# Statistical aggregation functions
STAT_FUNCTIONS = ['mean', 'std', 'min', 'max']


# =============================================================================
# SECTION 2: Data Processing Functions
# =============================================================================

def get_nth_element(data_list: list, n: int) -> Tuple[int, float]:
    """
    Find the index and value of the nth smallest element in a list.
    
    Args:
        data_list: List of numeric values
        n: Position to find (1-indexed)
        
    Returns:
        Tuple of (index_in_original_list, value)
    """
    sorted_list = sorted(data_list)
    nth_value = sorted_list[n - 1]
    return data_list.index(nth_value), nth_value


def get_distance_matrix(df: pd.DataFrame, df_features: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate pairwise Euclidean distance matrix for all points.
    
    Uses scipy.spatial.distance.cdist for efficient distance computation.
    
    Args:
        df: Original DataFrame with 'outlier' column
        df_features: DataFrame containing only feature columns (no labels)
        
    Returns:
        DataFrame with distance matrix where:
        - Columns 0 to n-1: distances to each point
        - Column 'outlier': ground truth labels from original dataset
    """
    # Calculate pairwise distances using Euclidean metric
    distance_matrix = cdist(df_features, df_features)
    
    # Convert to DataFrame with column names as point indices
    df_distance = pd.DataFrame(distance_matrix, columns=range(len(df)))
    
    # Attach ground truth labels
    df_distance['outlier'] = df['outlier']
    
    return df_distance


def initialize_knn_indices(df_distance: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate the k-nearest neighbor indices for each point.
    
    Computes indices of 1st, 10th, and 100th nearest neighbors, excluding
    self-distance (distance to itself).
    
    Args:
        df_distance: Distance matrix DataFrame from get_distance_matrix()
        
    Returns:
        DataFrame with additional columns: '1th', '10th', '100th' containing
        neighbor point indices
    """
    # Initialize columns for nearest neighbor indices
    df_distance['1th'] = 0
    df_distance['10th'] = 0
    df_distance['100th'] = 0
    
    pbar = tqdm(total=len(df_distance), desc="Computing KNN indices")
    
    # For each point
    for index, row in df_distance.iterrows():
        # Extract distances to all other points (convert to float)
        distances = [float(i) for i in row.tolist()[0:-4]]
        
        # Remove self-distance (distance from point to itself)
        distances.pop(index)
        
        # Find k-th nearest neighbors
        df_distance.loc[index, '1th'] = get_nth_element(distances, 1)[0]
        df_distance.loc[index, '10th'] = get_nth_element(distances, 10)[0]
        df_distance.loc[index, '100th'] = get_nth_element(distances, 100)[0]
        
        pbar.update(1)
    
    pbar.close()
    return df_distance


# =============================================================================
# SECTION 3: Density Analysis Functions
# =============================================================================

def compute_density_by_radius(df: pd.DataFrame, df_features: pd.DataFrame,
                             df_distance: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate local density for each point at multiple radius thresholds.
    
    Density is measured as the number of neighbors within each radius:
    - R1%, R5%, R10%, R20%, R25% of the maximum pairwise distance
    
    This provides a multi-scale density estimate useful for detecting
    outliers that may appear at different density scales.
    
    Args:
        df: Original DataFrame with 'outlier' column
        df_features: DataFrame with feature columns only
        df_distance: Distance matrix from get_distance_matrix()
        
    Returns:
        DataFrame with density columns added for each radius
    """
    # Calculate pairwise distances matrix
    distance_matrix = cdist(df_features, df_features)
    
    # Find maximum distance in dataset
    max_distance = np.max(pdist(df_features))
    
    # Initialize density columns
    for label in DENSITY_RADIUS_LABELS:
        df_distance[label] = 0
    
    # Calculate density for each radius percentage
    for radius_pct, radius_label in zip(DENSITY_RADII, DENSITY_RADIUS_LABELS):
        # Convert percentage to actual radius value
        radius = radius_pct * max_distance
        
        # Count neighbors within radius for each point
        densities = np.zeros(len(df_features))
        for i in range(len(df_features)):
            # Count points within radius (excluding self)
            densities[i] = np.sum(distance_matrix[i] <= radius) - 1
        
        df_distance[radius_label] = densities
    
    return df_distance


def add_outlier_labels(df: pd.DataFrame, df_features: pd.DataFrame,
                      df_distance: pd.DataFrame) -> pd.DataFrame:
    """
    Add index labels for visualization to outliers and isolated inliers.
    
    Labels are added based on density ranking (R25% radius).
    
    Args:
        df: Original DataFrame
        df_features: Feature columns only
        df_distance: Distance matrix with density columns
        
    Returns:
        DataFrame with 'label' column containing indices for visualization
    """
    # Initialize label column
    df['label'] = ''
    df_distance['label'] = ''
    
    # Sort by density (lowest density points appear first)
    df_sorted = sort_dataframe_by_column(df_distance, 'R25%')
    
    # Add labels for outliers
    outliers_df = df_features.iloc[df_sorted[(df_sorted['outlier'] == 'yes')].index.tolist()]
    for index, _ in outliers_df.iterrows():
        df.loc[index, 'label'] = index
        df_distance.loc[index, 'label'] = index
    
    # Add labels for inliers
    inliers_df = df_features.iloc[df_sorted[(df_sorted['outlier'] == 'no')].index.tolist()]
    for index, _ in inliers_df.iterrows():
        df.loc[index, 'label'] = index
        df_distance.loc[index, 'label'] = index
    
    return df


def sort_dataframe_by_column(dataframe: pd.DataFrame, column_name: str) -> pd.DataFrame:
    """
    Sort DataFrame by column and add position rank.
    
    Args:
        dataframe: DataFrame to sort
        column_name: Column to sort by
        
    Returns:
        Sorted DataFrame with additional 'position' column containing rank
    """
    sorted_df = dataframe.sort_values(by=[column_name])
    sorted_df['position'] = sorted_df.groupby(column_name, sort=False).ngroup() + 1
    return sorted_df


def print_density_summary(dataset_name: str, df_distance: pd.DataFrame) -> None:
    """
    Print comprehensive density statistics for the dataset.
    
    Reports mean, std, min, max density values for:
    - All points combined
    - Inliers only
    - Outliers only
    
    Args:
        dataset_name: Name of the dataset
        df_distance: Distance matrix with density columns
    """
    print(f'\n{"="*70}')
    print(f'Dataset: {dataset_name}')
    print(f'{"="*70}')
    
    # Print statistics for all points
    print('\nALL INSTANCES')
    print('Radius / Mean / Std / Min / Max')
    for label in DENSITY_RADIUS_LABELS:
        mean_val = round(df_distance[label].mean(), 2)
        std_val = round(df_distance[label].std(), 2)
        min_val = round(df_distance[label].min(), 2)
        max_val = round(df_distance[label].max(), 2)
        print(f'{label}: {mean_val} / {std_val} / {min_val} / {max_val}')
    
    # Print statistics for inliers
    print('\nINLIERS')
    print('Radius / Mean / Std / Min / Max')
    inliers = df_distance.query("outlier == 'no'")
    for label in DENSITY_RADIUS_LABELS:
        mean_val = round(inliers[label].mean(), 2)
        std_val = round(inliers[label].std(), 2)
        min_val = round(inliers[label].min(), 2)
        max_val = round(inliers[label].max(), 2)
        print(f'{label}: {mean_val} / {std_val} / {min_val} / {max_val}')
    
    print(inliers.sort_values(by=DENSITY_RADIUS_LABELS)[DENSITY_RADIUS_LABELS])
    
    # Print statistics for outliers
    print('\nOUTLIERS')
    print('Radius / Mean / Std / Min / Max')
    outliers = df_distance.query("outlier == 'yes'")
    for label in DENSITY_RADIUS_LABELS:
        mean_val = round(outliers[label].mean(), 2)
        std_val = round(outliers[label].std(), 2)
        min_val = round(outliers[label].min(), 2)
        max_val = round(outliers[label].max(), 2)
        print(f'{label}: {mean_val} / {std_val} / {min_val} / {max_val}')
    
    print(outliers.sort_values(by=DENSITY_RADIUS_LABELS)[DENSITY_RADIUS_LABELS])
    print()


def format_for_article(dataframe: pd.DataFrame, limit: int = 10) -> Tuple[List[int], List[int]]:
    """
    Extract and format data for publication, organizing by density ranking.
    
    Returns indices of most relevant inliers and outliers for visualization.
    
    Args:
        dataframe: Distance matrix DataFrame with density columns
        limit: Maximum number of instances to return per class
        
    Returns:
        Tuple of (inlier_indices, outlier_indices) to display
    """
    outliers_df = dataframe.query("outlier == 'yes'")
    inliers_df = dataframe.query("outlier == 'no'")
    
    # Sort inliers and outliers by density
    inliers_sorted = sort_dataframe_by_column(inliers_df, 'R25%')
    outliers_sorted = sort_dataframe_by_column(outliers_df, 'R25%')
    
    # Extract top instances (limited by outlier count or user limit)
    max_show = len(outliers_df) if len(outliers_df) <= limit else limit
    
    print('\nINLIERS - Density Ranking')
    print(inliers_sorted[DENSITY_RADIUS_LABELS + ['position']].head(max_show))
    inlier_indices = inliers_sorted.head(max_show).index.tolist()
    
    print('\nOUTLIERS - Density Ranking')
    print(outliers_sorted[DENSITY_RADIUS_LABELS + ['position']].head(max_show))
    outlier_indices = outliers_sorted.head(max_show).index.tolist()
    
    return inlier_indices, outlier_indices


def set_display_labels(dataframe: pd.DataFrame, indices_to_show: List[int],
                      instance_type: str) -> None:
    """
    Set visualization labels for specific instances.
    
    Clears labels for instances not in the provided list.
    
    Args:
        dataframe: DataFrame to modify
        indices_to_show: List of indices to keep labels for
        instance_type: Type of instances ('yes' for outliers, 'no' for inliers)
    """
    for index, row in dataframe.iterrows():
        if index not in indices_to_show and row['outlier'] == instance_type:
            dataframe.loc[index, 'label'] = ''


# =============================================================================
# SECTION 4: Dimensionality Reduction Functions
# =============================================================================

def apply_pca_projection(df_features: pd.DataFrame, n_components: int) -> np.ndarray:
    """
    Apply Principal Component Analysis (PCA) for dimensionality reduction.
    
    PCA finds orthogonal directions of maximum variance in the data.
    Useful for understanding dominant patterns and variance structure.
    
    Args:
        df_features: Feature matrix (samples × features)
        n_components: Number of dimensions to project to (typically 2 or 3)
        
    Returns:
        Projected data matrix (samples × n_components)
    """
    pca = PCA(n_components=n_components)
    return pca.fit_transform(df_features)


def apply_tsne_projection(df_features: pd.DataFrame, n_components: int) -> np.ndarray:
    """
    Apply t-Distributed Stochastic Neighbor Embedding (t-SNE).
    
    t-SNE preserves local structure and neighborhood relationships,
    making it excellent for discovering clusters and outliers.
    
    Args:
        df_features: Feature matrix (samples × features)
        n_components: Number of dimensions to project to (typically 2 or 3)
        
    Returns:
        Projected data matrix (samples × n_components)
    """
    tsne = TSNE(n_components=n_components, random_state=0)
    return tsne.fit_transform(df_features)


def apply_umap_projection(df_features: pd.DataFrame, n_components: int) -> np.ndarray:
    """
    Apply Uniform Manifold Approximation and Projection (UMAP).
    
    UMAP preserves both local and global structure while being more
    scalable than t-SNE. Often faster with better global structure.
    
    Args:
        df_features: Feature matrix (samples × features)
        n_components: Number of dimensions to project to (typically 2 or 3)
        
    Returns:
        Projected data matrix (samples × n_components)
    """
    umap_proj = UMAP(n_components=n_components, init='random', random_state=0)
    umap_proj.fit(df_features)
    return umap_proj.transform(df_features)


def get_projection_method(projection_class) -> callable:
    """
    Get the appropriate projection function for a given method class.
    
    Args:
        projection_class: Projection method class (PCA, TSNE, or UMAP)
        
    Returns:
        Projection function
    """
    projection_map = {
        PCA: apply_pca_projection,
        TSNE: apply_tsne_projection,
        UMAP: apply_umap_projection,
    }
    return projection_map.get(projection_class, None)


# =============================================================================
# SECTION 5: Visualization Functions
# =============================================================================

def plot_2d_projections(dataframe: pd.DataFrame, dataset_name: str,
                       output_dir: str = None, drive_path: str = None) -> None:
    """
    Generate 2D scatter plots for all dimensionality reduction methods.
    
    Creates scatter plots for all pairwise combinations of dimensions.
    Saves images and displays interactively using Plotly.
    
    Args:
        dataframe: DataFrame with features and labels
        dataset_name: Name of dataset (for titling and filenames)
        output_dir: Local directory to save images
        drive_path: Optional Google Drive path for backup
    """
    # Extract features (exclude 'label' and 'outlier' columns)
    df_features = dataframe.drop(columns=['label', 'outlier'])
    
    # Apply each projection method
    for projection_method in PROJECTION_METHODS:
        method_name = projection_method.__name__
        print(f"\nGenerating {method_name} projections...")
        
        # Get projection function and apply it
        projection_func = get_projection_method(projection_method)
        components = projection_func(df_features, DEFAULT_2D_DIMENSIONS)
        
        # Generate all pairwise 2D projections
        for i in range(DEFAULT_2D_DIMENSIONS):
            for j in range(DEFAULT_2D_DIMENSIONS):
                if i != j:  # Skip diagonal (same dimension)
                    # Create scatter plot
                    fig = px.scatter(
                        components,
                        x=i, y=j,
                        title=f'{method_name} ({dataset_name})',
                        hover_name=dataframe.index,
                        text=dataframe['label'],
                        color=dataframe['outlier'],
                        color_discrete_map={'yes': 'red', 'no': 'blue'}
                    )
                    
                    # Update layout and traces
                    fig.update_layout(
                        xaxis_title=f"Component {i+1}",
                        yaxis_title=f"Component {j+1}",
                        font=dict(size=14),
                        width=1000,
                        height=800
                    )
                    fig.update_traces(
                        textposition='top center',
                        textfont_size=10,
                        marker=dict(size=8)
                    )
                    
                    # Display in notebook
                    fig.show()
                    
                    # Save to local directory
                    if output_dir:
                        os.makedirs(output_dir, exist_ok=True)
                        filename = f'{dataset_name}_{method_name}_{i}_{j}.png'
                        filepath = os.path.join(output_dir, filename)
                        fig.write_image(filepath, scale=FIGURE_SCALE)
                    
                    # Save to Google Drive if path provided
                    if drive_path:
                        os.makedirs(drive_path, exist_ok=True)
                        filename = f'{dataset_name}_{method_name}_{i}_{j}.png'
                        filepath = os.path.join(drive_path, filename)
                        fig.write_image(filepath, scale=FIGURE_SCALE)


def plot_3d_projections(dataframe: pd.DataFrame, dataset_name: str,
                       output_dir: str = None, drive_path: str = None) -> None:
    """
    Generate 3D scatter plots for all dimensionality reduction methods.
    
    Creates interactive 3D scatter plots for better understanding of
    data structure in three dimensions.
    
    Args:
        dataframe: DataFrame with features and labels
        dataset_name: Name of dataset (for titling and filenames)
        output_dir: Local directory to save images
        drive_path: Optional Google Drive path for backup
    """
    # Extract features
    df_features = dataframe.drop(columns=['label', 'outlier'])
    
    # Apply each projection method
    for projection_method in PROJECTION_METHODS:
        method_name = projection_method.__name__
        print(f"\nGenerating {method_name} 3D projections...")
        
        # Get projection function and apply it
        projection_func = get_projection_method(projection_method)
        components = projection_func(df_features, DEFAULT_3D_DIMENSIONS)
        
        # Create 3D scatter plot
        fig = px.scatter_3d(
            components, x=0, y=1, z=2,
            title=f'{method_name} 3D ({dataset_name})',
            hover_name=dataframe.index,
            text=dataframe['label'],
            color=dataframe['outlier'],
            color_discrete_map={'yes': 'red', 'no': 'blue'}
        )
        
        # Update layout and traces
        fig.update_layout(font=dict(size=12))
        fig.update_traces(
            textposition='top center',
            textfont_size=10,
            marker=dict(size=6)
        )
        
        # Display in notebook
        fig.show()
        
        # Save to local directory
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
            filename = f'{dataset_name}_{method_name}_3D.png'
            filepath = os.path.join(output_dir, filename)
            fig.write_image(filepath, scale=FIGURE_SCALE)
        
        # Save to Google Drive if path provided
        if drive_path:
            os.makedirs(drive_path, exist_ok=True)
            filename = f'{dataset_name}_{method_name}_3D.png'
            filepath = os.path.join(drive_path, filename)
            fig.write_image(filepath, scale=FIGURE_SCALE)


# =============================================================================
# SECTION 6: Image Processing Functions
# =============================================================================

def crop_projection_images(directory: str, dimension_coords: str) -> np.ndarray:
    """
    Crop projection image to remove axis labels and whitespace.
    
    Uses predefined crop coordinates to extract relevant visualization area.
    
    Args:
        directory: Directory containing images
        dimension_coords: Dimension coordinate string (e.g., '0_1')
        
    Returns:
        Cropped image as numpy array
    """
    # Get crop coordinates for this dimension combination
    if dimension_coords not in CROP_COORDINATES:
        raise ValueError(f"No crop coordinates for {dimension_coords}")
    
    coord = CROP_COORDINATES[dimension_coords]
    
    # Read and crop image
    img = cv2.imread(os.path.join(directory, f'{dimension_coords}.png'))
    cropped = img[coord[0][0]:coord[0][1], coord[1][0]:coord[1][1]]
    
    return cropped


def resize_images_to_standard(images: List[np.ndarray], width: int = DEFAULT_IMAGE_WIDTH,
                             height: int = DEFAULT_IMAGE_HEIGHT) -> List[Image.Image]:
    """
    Resize all images to standard dimensions for consistent composition.
    
    Args:
        images: List of numpy arrays (images)
        width: Target width in pixels
        height: Target height in pixels
        
    Returns:
        List of PIL Image objects resized to (width, height)
    """
    resized_images = []
    
    for img_array in images:
        # Convert numpy array to PIL Image if needed
        if isinstance(img_array, np.ndarray):
            img = Image.fromarray(cv2.cvtColor(img_array, cv2.COLOR_BGR2RGB))
        else:
            img = img_array
        
        # Resize to standard dimensions
        img_resized = img.resize((width, height))
        resized_images.append(img_resized)
    
    return resized_images


def compose_image_grid_2x2(images: List[np.ndarray], output_path: str) -> np.ndarray:
    """
    Compose 4 images into a 2×2 grid layout.
    
    Arranges images as:
    [images[0] images[1]]
    [images[2] images[3]]
    
    Args:
        images: List of 4 numpy arrays (images)
        output_path: Path to save composed image
        
    Returns:
        Composed image as numpy array
    """
    # Vertical concatenations (rows)
    row1 = np.concatenate((images[0], images[1]), axis=1)
    row2 = np.concatenate((images[2], images[3]), axis=1)
    
    # Horizontal concatenation (final grid)
    composed = np.concatenate((row1, row2), axis=0)
    
    # Save result
    pil_image = Image.fromarray(composed)
    pil_image.save(output_path)
    
    return composed


def compose_image_grid_3x3(images: List[np.ndarray], output_path: str) -> np.ndarray:
    """
    Compose 9 images into a 3×3 grid layout.
    
    Args:
        images: List of 9 numpy arrays (images)
        output_path: Path to save composed image
        
    Returns:
        Composed image as numpy array
    """
    # Vertical concatenations (rows)
    row1 = np.concatenate((images[0], images[1], images[2]), axis=1)
    row2 = np.concatenate((images[3], images[4], images[5]), axis=1)
    row3 = np.concatenate((images[6], images[7], images[8]), axis=1)
    
    # Horizontal concatenation (final grid)
    composed = np.concatenate((row1, row2, row3), axis=0)
    
    # Save result
    pil_image = Image.fromarray(composed)
    pil_image.save(output_path)
    
    return composed


def compose_image_row(images: List[np.ndarray], output_path: str) -> np.ndarray:
    """
    Compose multiple images into a horizontal row.
    
    Args:
        images: List of numpy arrays (images)
        output_path: Path to save composed image
        
    Returns:
        Composed image as numpy array
    """
    # Concatenate all images horizontally
    composed = np.concatenate(images, axis=1)
    
    # Save result
    pil_image = Image.fromarray(composed)
    pil_image.save(output_path)
    
    return composed


def process_projection_images(dataset_dir: str, dataset_name: str,
                             output_dir: str, drive_path: str = None) -> None:
    """
    Process, crop, and compose projection images for publication.
    
    Main image processing pipeline that:
    1. Finds all projection images
    2. Crops to remove axis labels
    3. Resizes to standard dimensions
    4. Composes into grids
    5. Saves results locally and to Google Drive
    
    Args:
        dataset_dir: Directory containing dataset images
        dataset_name: Name of dataset
        output_dir: Local output directory
        drive_path: Optional Google Drive backup path
    """
    projection_methods = ['PCA', 'TSNE', 'UMAP']
    final_images = []
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Process each projection method
    for method in projection_methods:
        print(f"Processing {method} images...")
        cropped_images = []
        
        # Find and crop all images for this method
        files = os.listdir(dataset_dir)
        for filename in sorted(files):
            # Check if image belongs to this projection method
            if filename.find(f'{dataset_name}_{method}') >= 0:
                # Extract dimension coordinates
                dimension_match = re.findall(r'\d_\d', filename)
                if len(dimension_match) > 0:
                    dim_coords = dimension_match[0]
                    
                    try:
                        # Crop image
                        img = cv2.imread(os.path.join(dataset_dir, filename))
                        
                        if dim_coords in CROP_COORDINATES:
                            coord = CROP_COORDINATES[dim_coords]
                            cropped = img[coord[0][0]:coord[0][1],
                                        coord[1][0]:coord[1][1]]
                        else:
                            cropped = img
                        
                        # Add white bars to remove x-axis if needed
                        if dim_coords not in ['0_1', '1_1']:
                            cv2.rectangle(cropped, pt1=(0, 600), pt2=(cropped.shape[1], 630),
                                        color=(255, 255, 255), thickness=-1)
                        
                        # Save cropped version
                        cv2.imwrite(os.path.join(dataset_dir, f'{dim_coords}.png'), cropped)
                        cropped_images.append(cropped)
                    
                    except Exception as e:
                        print(f"Error processing {filename}: {e}")
        
        # Resize all images to standard size
        if cropped_images:
            resized = resize_images_to_standard(cropped_images)
            
            # Store the main dimension image (0_1) for final composition
            if len(resized) > 1:
                final_images.append(resized[1])
            
            # Compose grid for this method
            if len(resized) == 4:
                grid_path = os.path.join(output_dir, f'FINAL_{dataset_name}_{method}.png')
                compose_image_grid_2x2([cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
                                       for img in resized], grid_path)
            elif len(resized) == 9:
                grid_path = os.path.join(output_dir, f'FINAL_{dataset_name}_{method}.png')
                compose_image_grid_3x3([cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
                                       for img in resized], grid_path)
    
    # Compose final row from all methods
    if len(final_images) == 3:
        final_path = os.path.join(output_dir, f'FINAL_{dataset_name}.png')
        final_arrays = [cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR) for img in final_images]
        compose_image_row(final_arrays, final_path)
        
        # Backup to Google Drive if path provided
        if drive_path:
            os.makedirs(drive_path, exist_ok=True)
            compose_image_row(final_arrays, os.path.join(drive_path, f'FINAL_{dataset_name}.png'))


# =============================================================================
# SECTION 7: Main Execution Pipeline
# =============================================================================

def process_dataset(dataset_path: str, dataset_name: str, output_dir: str,
                   drive_path: str = None, generate_3d: bool = False) -> None:
    """
    Complete pipeline for dataset analysis and visualization.
    
    Orchestrates full workflow:
    1. Load dataset from CSV
    2. Extract features and calculate distance matrix
    3. Compute local density estimates
    4. Generate dimensionality reduction projections
    5. Create publication-ready visualizations
    6. Process and compose images
    
    Args:
        dataset_path: Path to CSV file
        dataset_name: Name of dataset (for display and filenames)
        output_dir: Local directory for output files
        drive_path: Optional Google Drive path for backup
        generate_3d: Whether to generate 3D projections
    """
    print(f"\n{'='*70}")
    print(f"Processing: {dataset_name}")
    print(f"{'='*70}")
    
    # ====================================================================
    # Step 1: Load and prepare data
    # ====================================================================
    print("\n[1/5] Loading dataset...")
    df = pd.read_csv(dataset_path)
    df_features = df.drop(columns=['outlier'])
    
    # ====================================================================
    # Step 2: Calculate distance matrix and density
    # ====================================================================
    print("[2/5] Computing distance matrix and density...")
    df_distance = get_distance_matrix(df, df_features)
    df_distance = compute_density_by_radius(df, df_features, df_distance)
    df = add_outlier_labels(df, df_features, df_distance)
    
    # ====================================================================
    # Step 3: Print density analysis
    # ====================================================================
    print("[3/5] Analyzing density...")
    print_density_summary(dataset_name, df_distance)
    inliers, outliers = format_for_article(df_distance)
    set_display_labels(df, inliers, 'no')
    
    # ====================================================================
    # Step 4: Generate projections
    # ====================================================================
    print("[4/5] Generating projections...")
    plot_2d_projections(df, dataset_name, output_dir, drive_path)
    if generate_3d:
        plot_3d_projections(df, dataset_name, output_dir, drive_path)
    
    # ====================================================================
    # Step 5: Process and compose images
    # ====================================================================
    print("[5/5] Processing images...")
    process_projection_images(output_dir, dataset_name, output_dir, drive_path)
    
    print(f"\n✓ Completed: {dataset_name}")

In [ ]:
def main():
    """
    Main execution pipeline for batch dataset visualization.
    
    Processes multiple datasets through complete analysis pipeline.
    """
    print("\n" + "="*70)
    print("Dataset Visualization and Analysis Pipeline")
    print("="*70)
    
    # Configure datasets to process
    datasets = [
        {'path': 'WPBC.csv', 'name': 'WPBC'},
        {'path': 'vertebral.csv', 'name': 'vertebral'},
        {'path': 'Wilt.csv', 'name': 'Wilt'},
        {'path': 'amazon.csv', 'name': 'amazon'},
    ]
    
    # Optional: Configure output directories
    local_output_dir = r'..\..\results\visualization_output'
    os.makedirs(local_output_dir, exist_ok=True)
    drive_path = None  # Set to Google Drive path if available
    
    # Process each dataset
    for dataset_config in datasets:
        try:
            process_dataset(
                dataset_path=os.sep.join([local_output_dir, dataset_config['path']]),
                dataset_name=dataset_config['name'],
                output_dir=os.path.join(local_output_dir, dataset_config['name']),
                drive_path=drive_path,
                generate_3d=False  # Set to True for 3D projections
            )
        except Exception as e:
            print(f"\n✗ Error processing {dataset_config['name']}: {e}")
            import traceback
            traceback.print_exc()
    
    print("\n" + "="*70)
    print("Pipeline Complete")
    print("="*70 + "\n")


if __name__ == "__main__":
    main()